In [1]:
# setup cv
import os
os.environ["OPENCV_VIDEOIO_MSMF_ENABLE_HW_TRANSFORMS"] = "0"

# datasets
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from huggingface_hub import HfApi

# control utils
from lerobot.utils.control_utils import init_keyboard_listener
from lerobot.utils.visualization_utils import _init_rerun
from lerobot.datasets.utils import hw_to_dataset_features
from lerobot.record import record_loop
from lerobot.utils.utils import log_say

# robot configs
from lerobot.cameras.opencv.configuration_opencv import OpenCVCameraConfig
from lerobot.cameras.configs import Cv2Rotation
from lerobot.teleoperators.so101_leader import SO101LeaderConfig, SO101Leader
from lerobot.robots.so101_follower import SO101FollowerConfig, SO101Follower

# utils
from dotenv import load_dotenv
from pprint import pprint

# my code
from scripts.utils.paths import CALIBS_DIR, REPO_ROOT, DATASETS_DIR, HF_NAME

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

C:\Users\jonathan\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

Recording Parameters:

In [2]:
NUM_EPISODES = 24
FPS = 30
EPISODE_TIME_SEC = 60 # how long each episode
RESET_TIME_SEC   = 10 # env reset time
TASK_DESCRIPTION = "move table leg from one hole to another"
REPO_NAME = 'so101_table_leg_move'
PUSH_TO_HUB = False

Login to HF if pushing:

In [3]:
if PUSH_TO_HUB:
    api = HfApi(token=os.getenv("HUGGINGFACE_TOKEN"))
    assert HF_NAME == api.whoami()['name']

Robot Config:

In [4]:
camera_config = {
    "wrist_cam": OpenCVCameraConfig(index_or_path=0, width=640, height=480, fps=30),
    "top_cam": OpenCVCameraConfig(index_or_path=1, width=640, height=480, fps=30, rotation=Cv2Rotation.NO_ROTATION),
}

robot_config = SO101FollowerConfig(
    port="COM7",
    id="so_101_follower",
    cameras = camera_config,
    calibration_dir = CALIBS_DIR
)

teleop_config = SO101LeaderConfig(
    port="COM8",
    id="so_101_leader",
    calibration_dir = CALIBS_DIR
)

robot = SO101Follower(robot_config)
teleop = SO101Leader(teleop_config)

Dataset build:

In [5]:
action_features = hw_to_dataset_features(robot.action_features, "action")
obs_features = hw_to_dataset_features(robot.observation_features, "observation")
dataset_features = {**action_features, **obs_features}
pprint(dataset_features)

{'action': {'dtype': 'float32',
            'names': ['shoulder_pan.pos',
                      'shoulder_lift.pos',
                      'elbow_flex.pos',
                      'wrist_flex.pos',
                      'wrist_roll.pos',
                      'gripper.pos'],
            'shape': (6,)},
 'observation.images.top_cam': {'dtype': 'video',
                                'names': ['height', 'width', 'channels'],
                                'shape': (480, 640, 3)},
 'observation.images.wrist_cam': {'dtype': 'video',
                                  'names': ['height', 'width', 'channels'],
                                  'shape': (480, 640, 3)},
 'observation.state': {'dtype': 'float32',
                       'names': ['shoulder_pan.pos',
                                 'shoulder_lift.pos',
                                 'elbow_flex.pos',
                                 'wrist_flex.pos',
                                 'wrist_roll.pos',
                          

Build Dataset:

In [6]:
dataset_path = DATASETS_DIR / REPO_NAME
if dataset_path.exists():
    dataset=LeRobotDataset(
        repo_id=f"{HF_NAME}/{REPO_NAME}",
        root=f"{DATASETS_DIR}\\{REPO_NAME}"
    )
else:
    dataset = LeRobotDataset.create(
        repo_id=f"{HF_NAME}/{REPO_NAME}",
        root=f"{DATASETS_DIR}\\{REPO_NAME}",
        fps=FPS,
        features=dataset_features,
        robot_type=robot.name,
        use_videos=True,
        image_writer_threads=4,
    )

Recording prep:

In [7]:
_, events = init_keyboard_listener()
_init_rerun(session_name="recording")

In [8]:
# connect to robots
robot.connect()
teleop.connect()

Recording Loop:

In [9]:
episode_idx = 0
while episode_idx < NUM_EPISODES and not events["stop_recording"]:
    log_say(f"Recording episode {episode_idx + 1} of {NUM_EPISODES}")

    record_loop(
        robot=robot,
        events=events,
        fps=FPS,
        teleop=teleop,
        dataset=dataset,
        control_time_s=EPISODE_TIME_SEC,
        single_task=TASK_DESCRIPTION,
        display_data=True,
    )

    # Reset the environment if not stopping or re-recording
    if not events["stop_recording"] and (episode_idx < NUM_EPISODES - 1 or events["rerecord_episode"]):
        log_say("Reset the environment")
        record_loop(
            robot=robot,
            events=events,
            fps=FPS,
            teleop=teleop,
            control_time_s=RESET_TIME_SEC,
            single_task=TASK_DESCRIPTION,
            display_data=True,
        )

    if events["rerecord_episode"]:
        log_say("Re-recording episode")
        events["rerecord_episode"] = False
        events["exit_early"] = False
        dataset.clear_episode_buffer()
        continue

    dataset.save_episode()
    episode_idx += 1

Right arrow key pressed. Exiting loop...
Escape key pressed. Stopping data recording...


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 333.17ba/s]


Right arrow key pressed. Exiting loop...
Left arrow key pressed. Exiting loop and rerecord the last episode...
Right arrow key pressed. Exiting loop...


In [ ]:
log_say("Stop recording")
robot.disconnect()
teleop.disconnect()
if PUSH_TO_HUB:
    dataset.push_to_hub()

KeyboardInterrupt: 

Escape key pressed. Stopping data recording...
